# 25 — `aro ask` Interface

Interactive interface for the fine-tuned ARO model, matching the `aro ask` CLI command.

The model handles two modes automatically:
- **Code generation** — when the user asks to write/create/build something, responds with ARO code
- **Knowledge Q&A** — when the user asks a question, answers from ARO language knowledge

**Model management:**
- Checks if the model is already downloaded before fetching
- Compares local version against HuggingFace to detect updates
- Asks the user before downloading or updating

**Input:**  `config.PREFERRED_MODEL_ID` + `config.build_system_prompt()`
**Usage:**  run cells top-to-bottom, then use the chat loop cell repeatedly

## Setup & Model Management

In [ ]:
import sys, importlib, re, subprocess, tempfile, json, hashlib
from pathlib import Path

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

MAX_TOKENS   = 1024
TEMPERATURE  = 0.3
TOP_P        = 0.9

print(f'Model ID: {PREFERRED_MODEL_ID}')
print(f'Release dir: {RELEASE_DIR}')

## Check model version & download

Before loading the model, check:
1. Is the model already downloaded locally?
2. Is the local version up-to-date with HuggingFace?
3. Ask the user before downloading or updating.

In [ ]:
def _get_hf_cache_path(model_id):
    """Return the local HuggingFace cache path for a model, or None if not cached."""
    try:
        from huggingface_hub import scan_cache_dir
        cache = scan_cache_dir()
        for repo in cache.repos:
            if repo.repo_id == model_id:
                # Return the most recent revision's snapshot path
                revisions = sorted(repo.revisions, key=lambda r: r.last_modified, reverse=True)
                if revisions:
                    return revisions[0].snapshot_path
    except Exception:
        pass
    return None


def _get_remote_commit(model_id, timeout=10):
    """Fetch the latest commit SHA from HuggingFace for comparison."""
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        info = api.model_info(model_id, timeout=timeout)
        return info.sha
    except Exception:
        return None


def _get_local_commit(model_id):
    """Get the commit SHA of the locally cached model."""
    try:
        from huggingface_hub import scan_cache_dir
        cache = scan_cache_dir()
        for repo in cache.repos:
            if repo.repo_id == model_id:
                revisions = sorted(repo.revisions, key=lambda r: r.last_modified, reverse=True)
                if revisions:
                    return revisions[0].commit_hash
    except Exception:
        pass
    return None


def check_model_status(model_id):
    """Check if model is downloaded and up-to-date.

    Returns (status, details) where status is one of:
      'ready'       — model is downloaded and up-to-date
      'update'      — model is downloaded but a newer version exists
      'missing'     — model is not downloaded
      'offline'     — model is downloaded but can't check for updates
    """
    local_path = _get_hf_cache_path(model_id)

    if local_path is None:
        # Not cached — check if it's a local directory path (e.g. release/aro-coder-4bit)
        release_path = RELEASE_DIR / 'aro-coder-4bit'
        if release_path.exists() and (release_path / 'config.json').exists():
            return 'ready', {'path': str(release_path), 'source': 'local_release'}
        return 'missing', {}

    # Model is cached — check for updates
    local_commit = _get_local_commit(model_id)
    remote_commit = _get_remote_commit(model_id)

    if remote_commit is None:
        return 'offline', {'path': str(local_path), 'local_commit': local_commit}

    if local_commit == remote_commit:
        return 'ready', {'path': str(local_path), 'commit': local_commit}

    return 'update', {
        'path': str(local_path),
        'local_commit': local_commit,
        'remote_commit': remote_commit,
    }


# ── Check model status ──────────────────────────────────────────────────────
status, details = check_model_status(PREFERRED_MODEL_ID)

if status == 'ready':
    print(f'Model is downloaded and up-to-date.')
    if 'commit' in details:
        print(f'  Commit: {details["commit"][:12]}')
    print(f'  Path: {details.get("path", "HF cache")}')
    _model_path = PREFERRED_MODEL_ID

elif status == 'offline':
    print(f'Model is downloaded (offline — cannot check for updates).')
    print(f'  Local commit: {details.get("local_commit", "?")[:12]}')
    print(f'  Path: {details["path"]}')
    _model_path = PREFERRED_MODEL_ID

elif status == 'update':
    print(f'A newer version of the model is available on HuggingFace.')
    print(f'  Local:  {details.get("local_commit", "?")[:12]}')
    print(f'  Remote: {details.get("remote_commit", "?")[:12]}')
    answer = input('\nDownload the updated model? [y/N]: ').strip().lower()
    if answer in ('y', 'yes'):
        print(f'Downloading {PREFERRED_MODEL_ID} (update)...')
        from huggingface_hub import snapshot_download
        snapshot_download(PREFERRED_MODEL_ID, force_download=True)
        print('Update complete.')
    else:
        print('Keeping current version.')
    _model_path = PREFERRED_MODEL_ID

elif status == 'missing':
    print(f'Model {PREFERRED_MODEL_ID} is not downloaded.')
    # Check if remote exists before asking
    if _hf_model_exists(PREFERRED_MODEL_ID):
        answer = input(f'\nDownload {PREFERRED_MODEL_ID} from HuggingFace? [y/N]: ').strip().lower()
        if answer in ('y', 'yes'):
            print(f'Downloading {PREFERRED_MODEL_ID}...')
            from huggingface_hub import snapshot_download
            snapshot_download(PREFERRED_MODEL_ID)
            print('Download complete.')
            _model_path = PREFERRED_MODEL_ID
        else:
            print('Download cancelled. Falling back to local release model or base model.')
            _release = RELEASE_DIR / 'aro-coder-4bit'
            if _release.exists() and (_release / 'config.json').exists():
                _model_path = str(_release)
                print(f'Using local release: {_model_path}')
            else:
                _model_path = MODEL_ID
                print(f'Using base model: {_model_path}')
    else:
        print(f'Model not found on HuggingFace either.')
        _release = RELEASE_DIR / 'aro-coder-4bit'
        if _release.exists() and (_release / 'config.json').exists():
            _model_path = str(_release)
            print(f'Using local release: {_model_path}')
        else:
            _model_path = MODEL_ID
            print(f'Using base model: {_model_path}')

print(f'\nWill load: {_model_path}')

## Load model

In [ ]:
load_fn, generate_fn, make_sampler_fn = ensure_mlx_lm()

print(f'Loading {_model_path}...')
model, tokenizer = load_fn(_model_path)
print('Model loaded.')

kb            = load_knowledge()
SYSTEM_PROMPT = build_system_prompt(kb)

print(f'\nSystem prompt ({len(SYSTEM_PROMPT)} chars):')
print('─' * 60)
print(SYSTEM_PROMPT[:600] + ('...' if len(SYSTEM_PROMPT) > 600 else ''))
print('─' * 60)

## Helpers

In [ ]:
def _generate(messages, max_tokens=MAX_TOKENS, temperature=TEMPERATURE):
    """Run inference on a messages list. Returns the assistant reply string."""
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    reply  = generate_fn(
        model, tokenizer,
        prompt  = tokens,
        max_tokens = max_tokens,
        sampler = make_sampler_fn(temp=temperature, top_p=TOP_P),
        verbose = False,
    )
    return reply.strip()


def extract_aro_blocks(text):
    """Return list of ARO code blocks from the response."""
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]


# Heuristic: a reply is "checkable" only if at least one ARO block contains a
# feature-set header `(name: activity) {`. Q&A replies often include short
# illustrative fragments that are valid ARO syntax in context but not a
# runnable program — running `aro check` on those produces a misleading FAIL.
_FEATURESET_HEADER = re.compile(r'\(\s*[\w\- ]+\s*:\s*[^)]+\)\s*\{', re.DOTALL)


def is_complete_program(blocks):
    return any(_FEATURESET_HEADER.search(b) for b in blocks)


def aro_check(code):
    """Run `aro check` on a snippet. Returns (True/False/None, output_text).
    None means aro was not found in PATH."""
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0, r.stdout + r.stderr
    except FileNotFoundError:
        return None, 'aro not found in PATH'
    except Exception as e:
        return None, str(e)


print('Helpers ready.')

## `aro ask` chat loop

Run this cell to start an interactive session (mirrors `aro ask` CLI behavior).

- Type a question about ARO → get a knowledge answer
- Type a request to write code → get ARO code (auto-validated with `aro check`)
- Type `quit` / `exit` / `q` to stop
- Type `reset` to clear history
- Type `check` to re-validate the last generated code

In [ ]:
history = [{'role': 'system', 'content': SYSTEM_PROMPT}]
last_reply = ''

print('aro ask  (quit/exit/q to stop · reset to clear · check to validate last code)')
print('=' * 70)

while True:
    try:
        user_input = input('\nYou: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n[Interrupted]')
        break

    if not user_input:
        continue

    if user_input.lower() in ('quit', 'exit', 'q'):
        print('Goodbye.')
        break

    if user_input.lower() == 'reset':
        history = [{'role': 'system', 'content': SYSTEM_PROMPT}]
        last_reply = ''
        print('[History cleared]')
        continue

    if user_input.lower() == 'check':
        blocks = extract_aro_blocks(last_reply)
        if not blocks:
            print('[No ARO code blocks found in last reply]')
        elif not is_complete_program(blocks):
            print('[Last reply contains only ARO snippets, not a complete program — aro check skipped]')
        else:
            code = '\n\n'.join(blocks)
            ok, output = aro_check(code)
            status = '✓ PASS' if ok is True else ('✗ FAIL' if ok is False else '? (aro not in PATH)')
            print(f'[aro check: {status}]')
            if output.strip():
                print(output.strip())
        continue

    history.append({'role': 'user', 'content': user_input})

    print('\nAssistant: ', end='', flush=True)
    try:
        reply = _generate(history)
    except Exception as e:
        print(f'[Error: {e}]')
        history.pop()  # remove the user turn on failure
        continue

    print(reply)
    last_reply = reply
    history.append({'role': 'assistant', 'content': reply})

    # Auto-validate only when the reply contains a complete ARO program.
    # Q&A replies often include illustrative fragments; running aro check
    # on a fragment would FAIL misleadingly, so skip it.
    blocks = extract_aro_blocks(reply)
    if blocks and is_complete_program(blocks):
        code = '\n\n'.join(blocks)
        ok, output = aro_check(code)
        if ok is True:
            print('\n[aro check: ✓ PASS]')
        elif ok is False:
            print(f'\n[aro check: ✗ FAIL]')
            if output.strip():
                print(output.strip())

## Single-turn quick test

Tests both modes: code generation (write) and knowledge Q&A (question).

In [ ]:
TEST_CASES = [
    ('code', 'Write a minimal ARO application that logs Hello World and exits.'),
    ('qa',   'What is the difference between Extract and Compute actions in ARO?'),
]

for mode, prompt in TEST_CASES:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]

    reply = _generate(messages)
    print(f'[{mode.upper()}] PROMPT:\n{prompt}\n')
    print(f'RESPONSE:\n{reply}\n')

    blocks = extract_aro_blocks(reply)
    if blocks and is_complete_program(blocks):
        code = '\n\n'.join(blocks)
        ok, output = aro_check(code)
        status = '✓ PASS' if ok is True else ('✗ FAIL' if ok is False else '? (aro not in PATH)')
        print(f'aro check: {status}')
        if output.strip():
            print(output.strip())
    elif blocks:
        # Fragments only — illustrative snippets in Q&A. Skip aro check.
        print('[aro check skipped: reply has ARO snippets but no complete program]')
    else:
        if mode == 'code':
            print('[WARNING: No ARO code blocks in code generation response]')
        else:
            print('[OK: Q&A response — no code block expected]')
    print('─' * 70)

## Batch probe

Runs a fixed set of prompts covering both code generation and knowledge Q&A.
Reports syntax pass rate for code prompts and answer quality for Q&A prompts.

In [ ]:
PROBES = [
    # Code generation probes — expect ```aro blocks, validated with aro check
    ('code', 'Write a minimal ARO Application-Start that starts an HTTP server.'),
    ('code', 'Write an ARO feature set that retrieves a user by ID from a repository and returns it as an OK response.'),
    ('code', 'Write an ARO for-each loop that logs each item in a list.'),
    ('code', 'Write an ARO feature set that validates a request body and returns a 400 status on failure.'),
    ('code', 'Write an ARO Application-Start that reads a file and logs its contents.'),
    # Knowledge Q&A probes — expect explanatory text, optionally with examples
    ('qa', 'How do I emit a custom event in ARO and handle it in another feature set?'),
    ('qa', 'Explain the difference between Extract and Compute in ARO.'),
    ('qa', 'How does the Publish action work in ARO? Give an example.'),
]

code_passed  = 0
code_failed  = 0
code_skipped = 0      # reply had only fragments, no complete program
qa_answered  = 0
qa_empty     = 0

print(f'Running {len(PROBES)} probes...\n')

for i, (mode, prompt) in enumerate(PROBES):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    reply  = _generate(messages)
    blocks = extract_aro_blocks(reply)

    if mode == 'code':
        if not blocks:
            status = '—  no code (skip)'
            code_skipped += 1
        elif not is_complete_program(blocks):
            status = '—  fragment (skip)'
            code_skipped += 1
        else:
            ok, _ = aro_check('\n\n'.join(blocks))
            if ok is True:
                status = '✓  pass'
                code_passed += 1
            elif ok is False:
                status = '✗  FAIL'
                code_failed += 1
            else:
                status = '?  (aro not in PATH)'
    else:  # qa
        # Q&A: check that there's meaningful explanatory text (not just code)
        text_only = re.sub(r'```.*?```', '', reply, flags=re.DOTALL).strip()
        if len(text_only) > 30:
            status = '✓  answered'
            qa_answered += 1
        else:
            status = '—  thin answer'
            qa_empty += 1

    print(f'  [{i+1:2d}] [{mode:4s}] {status:<22}  {prompt[:60]}')

total_code = code_passed + code_failed
print(f'\nCode probes:  {code_passed}/{total_code} syntax pass' +
      (f' ({100*code_passed/total_code:.0f}%)' if total_code else ''))
if code_skipped:
    print(f'              {code_skipped} skipped (reply had no complete program)')
print(f'Q&A probes:   {qa_answered}/{qa_answered+qa_empty} answered' +
      (f' ({100*qa_answered/(qa_answered+qa_empty):.0f}%)' if (qa_answered+qa_empty) else ''))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9', 'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
from pathlib import Path
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '23_chat.png'

total_code = code_passed + code_failed
total_qa   = qa_answered + qa_empty

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# ── Left: Code probe results ─────────────────────────────────────────
_code_vals = [code_passed, code_failed]
_code_labels = [f'Pass\n({code_passed})', f'Fail\n({code_failed})']
_code_colors = ['#2ecc71', '#e74c3c']
if total_code > 0:
    wedges, _, autotexts = ax1.pie(
        _code_vals, labels=_code_labels, colors=_code_colors,
        autopct='%1.0f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    )
    for t in autotexts: t.set_fontsize(11); t.set_fontweight('bold')
    ax1.add_artist(plt.Circle((0, 0), 0.5, fc='white'))
    ax1.text(0, 0, f'{100*code_passed/total_code:.0f}%', ha='center', va='center',
             fontsize=16, fontweight='bold')
else:
    ax1.text(0.5, 0.5, 'no code probes', ha='center', va='center', transform=ax1.transAxes)
ax1.set_title('Code Probes (aro check)', fontsize=12, fontweight='bold')

# ── Right: Q&A probe results ─────────────────────────────────────────
_qa_vals = [qa_answered, qa_empty]
_qa_labels = [f'Answered\n({qa_answered})', f'Thin\n({qa_empty})']
_qa_colors = ['#3498db', '#f39c12']
if total_qa > 0:
    wedges, _, autotexts = ax2.pie(
        _qa_vals, labels=_qa_labels, colors=_qa_colors,
        autopct='%1.0f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    )
    for t in autotexts: t.set_fontsize(11); t.set_fontweight('bold')
    ax2.add_artist(plt.Circle((0, 0), 0.5, fc='white'))
    ax2.text(0, 0, f'{100*qa_answered/total_qa:.0f}%', ha='center', va='center',
             fontsize=16, fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'no Q&A probes', ha='center', va='center', transform=ax2.transAxes)
ax2.set_title('Q&A Probes', fontsize=12, fontweight='bold')

fig.suptitle(
    f'Chat Probes — {total_code + total_qa} probes  ·  '
    f'{code_passed}/{total_code} code pass  ·  {qa_answered}/{total_qa} Q&A answered',
    fontsize=12, fontweight='bold', y=1.04,
)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')
